# Module 5: Observability, Files & System Ops

**Concepts:** `pathlib` · regex · structured logging · `subprocess`

**Audience:** Read manifests, grep logs, emit JSON logs for Loki/ELK, wrap shell tools safely.

---

**Rhythm:** markdown → code → run.


## 1. `pathlib.Path` — paths without string pain

Cross-platform paths, join with `/`, read text, check exists.


In [ ]:
from pathlib import Path

base = Path(".")
readme = base / "gemini" / "generate_modules_2_8.py"
print("exists:", readme.exists())
print("name:", readme.name, "suffix:", readme.suffix)


## 2. Read and write files

Use `with open` or `Path.read_text()` / `write_text()` for UTF-8 configs.


In [ ]:
from pathlib import Path

tmp = Path("gemini/_lab_config.txt")
tmp.write_text("replicas=3\nenv=staging\n", encoding="utf-8")
print(tmp.read_text(encoding="utf-8"))
tmp.unlink(missing_ok=True)


## 3. Regex — extract from logs

`re.search`, groups, `re.compile` for repeated patterns (HTTP codes, pod IDs).


In [ ]:
import re

line = "2026-03-28 ERROR pod api-7fd9 OOMKilled exit=137"
pat = re.compile(r"pod (?P<pod>\S+) (?P<reason>\S+)")
m = pat.search(line)
if m:
    print(m.group("pod"), m.group("reason"))


## 4. Structured logging

Plain `print` does not scale — use **logging** with levels; JSON formatter friendly to OpenTelemetry/Loki.


In [ ]:
import json
import logging

logger = logging.getLogger("platform")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler()
logger.addHandler(handler)


def log_json(level: str, msg: str, **fields):
    payload = {"level": level, "msg": msg, **fields}
    logger.info(json.dumps(payload))


log_json("info", "rollout started", cluster="ncs57", ns="iim-lhp")


## 5. `subprocess` — wrap CLI tools

Prefer list args (no shell injection): `["kubectl", "get", "pods", "-n", ns]`.

**This cell is dry-run only** — prints the command instead of calling kubectl on your machine.


In [ ]:
import shlex
from subprocess import list2cmdline

ns = "iim-lhp"
cmd = ["kubectl", "get", "pods", "-n", ns, "-o", "wide"]
print("would run:", list2cmdline(cmd))
print("argv:", cmd)


## 6. Capstone — parse config file + regex on log snippet

Combine pathlib + regex like a small triage script.


In [ ]:
from pathlib import Path
import re

cfg_text = "level=ERROR\ncount=3\n"
cfg = dict(line.split("=", 1) for line in cfg_text.strip().split("\n"))
log = "ERROR connection timeout host=10.0.0.5"
host = re.search(r"host=(?P<h>\S+)", log)
print("cfg", cfg, "host", host.group("h") if host else None)


---
# Exercises


**Ex 1** — Path `p = Path('gemini')`; print `.is_dir()` and count `*.ipynb` with `.glob`.


In [ ]:
# TODO


**Ex 2** — Regex extract HTTP status from `'GET /health 503 12ms'`.


In [ ]:
# TODO


**Ex 3** — Log one JSON line with keys `event`, `namespace`, `pod` using `log_json` pattern from above.


In [ ]:
# TODO


**Ex 4** — Build safe argv list for `helm status release -n ns` (no shell=True).


In [ ]:
# TODO


---
# Solutions


In [ ]:
from pathlib import Path

p = Path("gemini")
print("is_dir", p.is_dir())
print(" notebooks", len(list(p.glob("*.ipynb"))))


In [ ]:
import re

line = "GET /health 503 12ms"
m = re.search(r"\b(\d{3})\b", line)
print(int(m.group(1)) if m else None)


In [ ]:
import json
import logging

logger = logging.getLogger("ex3")
logger.handlers.clear()
logger.addHandler(logging.StreamHandler())
logger.info(json.dumps({"event": "restart", "namespace": "iim-lhp", "pod": "api-0"}))


In [ ]:
release, ns = "myrel", "iim-lhp"
cmd = ["helm", "status", release, "-n", ns]
print(cmd)


**Next:** Module 6 — HTTP clients and APIs.
